In [ ]:
!pip install pandas mysql-connector-python

import pandas as pd
import mysql.connector
from mysql.connector import Error
import math
import unicodedata

# Configurações do Banco de Dados
DB_CONFIG = {
    'host': 'localhost',
    'database': 'mydb', # Altere para o nome do seu schema
    'user': 'root',                  # Altere para seu usuário
    'password': '1234'          # Altere para sua senha
}


def clean_value(val):
    """Limpa valores NaN do pandas para o MySQL (converte para None)"""
    if pd.isna(val):
        return None
    return str(val).strip()

def insert_data():
    try:
        # 1. Conecta ao banco de dados MySQL
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor()
        
        # 2. Carrega o CSV
        csv_file = 'C:\\Users\\NARCI\\OneDrive\\Desktop\\Univesp - PI\\CAPES - IES - Brasil (GEOREFERENCIADA) V7.csv'
        df = pd.read_csv(csv_file)
        total_linhas = len(df)

        # Função para remover acentos
        def remover_acentos(texto):
            return ''.join(
                c for c in unicodedata.normalize('NFD', texto)
                if unicodedata.category(c) != 'Mn'
            )

        # Aplicar a função Remover Acentos na coluna desejada
        df['IES - SIGLA'] = df['IES - SIGLA'].apply(remover_acentos)
        df['IES - NOME'] = df['IES - NOME'].apply(remover_acentos)
        df['Endereço'] = df['Endereço'].apply(remover_acentos)
        df['Município'] = df['Município'].apply(remover_acentos)
        df['Status Jurídico'] = df['Status Jurídico'].apply(remover_acentos)
        df['Áreas de pesquisa'] = df['Áreas de pesquisa'].apply(remover_acentos)
        df['Programa de Pós-graduação'] = df['Programa de Pós-graduação'].apply(remover_acentos)
        df['Área Conhecimento'] = df['Área Conhecimento'].apply(remover_acentos)
        df['Especialização'] = df['Especialização'].apply(remover_acentos)
        df['Mestrado'] = df['Mestrado'].apply(remover_acentos)
        df['Doutorado'] = df['Doutorado'].apply(remover_acentos)
        print("Remoção de acentos concluída com sucesso!")

        # Substituir ç por c e ç por C
        df = df.replace(to_replace='ç', value='c', regex=True)
        df = df.replace(to_replace='Ç', value='C', regex=True)
        print("Remoção de Ç concluída com sucesso!")

        # Substitui vírgulas por pontos e converte para float, se necessário
        df['Latitude'] = df['Latitude'].astype(str).str.replace(',', '.').astype(float)
        df['Longitude'] = df['Longitude'].astype(str).str.replace(',', '.').astype(float)
        print("Substituição de vírgulas por pontos concluída com sucesso!")
        
        # Dicionários de cache para evitar múltiplas inserções e guardar IDs gerados (Chaves Estrangeiras)
        cache_estados = {}
        cache_municipios = {}
        cache_ies = {}
        cache_areas_conhecimento = {}
        cache_programas_pg = {}
        estados = {"AC": "Acre", "AL": "Alagoas", "AP": "Amapa", "AM": "Amazonas", "BA": "Bahia", "CE": "Ceara", "DF": "Distrito Federal", "ES": "Espirito Santo", "GO": "Goias", "MA": "Maranhao", "MT": "Mato Grosso", "MS": "Mato Grosso do Sul", "MG": "Minas Gerais", "PA": "Para", "PB": "Paraíba", "PR": "Parana", "PE": "Pernambuco", "PI": "Piaui", "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte", "RS": "Rio Grande do Sul", "RO": "Rondonia", "RR": "Roraima", "SC": "Santa Catarina", "SP": "Sao Paulo", "SE": "Sergipe", "TO": "Tocantins"}

        print("Iniciando a inserção de dados...")

        # 3. Itera sobre cada linha do DataFrame
        for index, row in df.iterrows():
            print(f"Processando linha {index + 1}/{total_linhas} - IES: {row['IES - NOME']}")
            # --- TABELA: estados ---
            uf = clean_value(row['UF'])
            if uf and uf not in cache_estados:
                #cursor.execute("INSERT INTO estados (sigla_estado) VALUES (%s)", (uf,))
                cursor.execute("INSERT INTO estados (sigla_estado, estado) VALUES (%s, %s)", (uf, estados.get(uf, 'Não Informado')))
                cache_estados[uf] = cursor.lastrowid
            estado_id = cache_estados.get(uf)

            # --- TABELA: municipios ---
            municipio = clean_value(row['Município'])
            chave_mun = f"{municipio}-{estado_id}"
            if municipio and chave_mun not in cache_municipios:
                cursor.execute("INSERT INTO municipios (municipio, estados_estado_id) VALUES (%s, %s)", 
                               (municipio, estado_id))
                cache_municipios[chave_mun] = cursor.lastrowid
            municipio_id = cache_municipios.get(chave_mun)

            # --- TABELA: instituicoes_ensino_superior ---
            sigla_ies = clean_value(row['IES - SIGLA'])
            nome_ies = clean_value(row['IES - NOME'])
            homepage = clean_value(row['Homepage'])
            status_juridico = clean_value(row['Status Jurídico'])
            
            if sigla_ies and sigla_ies not in cache_ies:
                cursor.execute("""
                    INSERT INTO instituicoes_ensino_superior (sigla_ies, nome_ies, homepage, status_juridico)
                    VALUES (%s, %s, %s, %s)
                """, (sigla_ies, nome_ies, homepage, status_juridico))
                ies_id = cursor.lastrowid
                cache_ies[sigla_ies] = ies_id
                
                # --- TABELA: enderecos_ies (Associado à IES inserida) ---
                logradouro = clean_value(row['Endereço'])
                lat = clean_value(row['Latitude'])
                lon = clean_value(row['Longitude'])
                # Trata formatação de lat/lon se vier com vírgula do CSV
                if lat: lat = lat.replace(',', '.')
                if lon: lon = lon.replace(',', '.')

                cursor.execute("""
                    INSERT INTO enderecos_ies (logradouro, latitude, longitude, instituicoes_ensino_superior_ies_id, municipios_municipio_id)
                    VALUES (%s, %s, %s, %s, %s)
                """, (logradouro, lat, lon, ies_id, municipio_id))
            
            ies_id = cache_ies.get(sigla_ies)

            # --- TABELA: areas_conhecimento ---
            area_conh = clean_value(row['Área Conhecimento'])
            if area_conh and area_conh not in cache_areas_conhecimento:
                cursor.execute("INSERT INTO areas_conhecimento (area_conhecimento) VALUES (%s)", (area_conh,))
                area_conh_id = cursor.lastrowid
                cache_areas_conhecimento[area_conh] = area_conh_id
            area_conh_id = cache_areas_conhecimento.get(area_conh)

            # Relacionamento N:M: areas_conhecimento_has_instituicoes_ensino_superior
            if area_conh_id and ies_id:
                try:
                    cursor.execute("""
                        INSERT IGNORE INTO areas_conhecimento_has_instituicoes_ensino_superior 
                        (areas_conhecimento_area_conhecimento_id, instituicoes_ensino_superior_ies_id)
                        VALUES (%s, %s)
                    """, (area_conh_id, ies_id))
                except:
                    pass # Ignora se já estiver relacionado

            # --- TABELA: programa_pos_graduacao ---
            prog_pg = clean_value(row['Programa de Pós-graduação'])
            insc_inicio = clean_value(row['Período do inscrição – Início'])
            insc_fim = clean_value(row['Período do inscrição – Fim'])
            
            chave_prog = f"{prog_pg}-{area_conh_id}"
            if prog_pg and chave_prog not in cache_programas_pg:
                # Formatar data se necessário (assumindo que o CSV está no padrão DD/MM/YY)
                # O ideal é tratar a string de data para o formato YYYY-MM-DD do MySQL
                if insc_inicio:
                    try: insc_inicio = pd.to_datetime(insc_inicio, format='%d/%m/%y').strftime('%Y-%m-%d')
                    except: pass
                if insc_fim:
                    try: insc_fim = pd.to_datetime(insc_fim, format='%d/%m/%y').strftime('%Y-%m-%d')
                    except: pass

                cursor.execute("""
                    INSERT INTO programa_pos_graduacao (programa_pg, areas_conhecimento_area_conhecimento_id, incricao_inicio, inscricao_fim)
                    VALUES (%s, %s, %s, %s)
                """, (prog_pg, area_conh_id, insc_inicio, insc_fim))
                cache_programas_pg[chave_prog] = cursor.lastrowid
            
            prog_pg_id = cache_programas_pg.get(chave_prog)

            # --- TABELA: areas_pesquisa ---
            area_pesquisa = clean_value(row['Áreas de pesquisa'])
            if area_pesquisa and prog_pg_id:
                cursor.execute("""
                    INSERT INTO areas_pesquisa (area_pesquisa, programa_pos_graduacao_programa_pg_id)
                    VALUES (%s, %s)
                """, (area_pesquisa, prog_pg_id))

            # --- TABELA: modalidades_pos_graduacao ---
            especializacao = clean_value(row['Especialização'])
            mestrado = clean_value(row['Mestrado'])
            doutorado = clean_value(row['Doutorado'])
            nota_mec = clean_value(row['Nota de Avaliação – Especialização – MEC'])
            nota_capes = clean_value(row['Nota de Avaliação – Pós-graduação – Capes'])

            modalidades = []
            if especializacao == 'SIM': modalidades.append('Especialização')
            if mestrado == 'SIM': modalidades.append('Mestrado')
            if doutorado == 'SIM': modalidades.append('Doutorado')
            
            for mod in modalidades:
                cursor.execute("""
                    INSERT INTO modalidades_pos_graduacao 
                    (modalidade_pos_graduacao, nota_espec_MEC, nota_mestrado_doutorado_Capes, programa_pos_graduacao_programa_pg_id)
                    VALUES (%s, %s, %s, %s)
                """, (mod, nota_mec if mod == 'Especialização' else None, nota_capes if mod in ['Mestrado', 'Doutorado'] else None, prog_pg_id))

        # Commita todas as alterações no banco de dados
        conn.commit()
        print("Dados importados com sucesso!")

    except Error as e:
        print(f"Erro ao acessar o MySQL: {e}")
        if 'conn' in locals() and conn.is_connected():
            conn.rollback() # Desfaz as alterações em caso de erro
    finally:
        if 'conn' in locals() and conn.is_connected():
            cursor.close()
            conn.close()
            print("Conexão ao MySQL encerrada.")

if __name__ == '__main__':
    insert_data()